<a href="https://colab.research.google.com/github/Anselemcrizy/Login-Register/blob/main/VIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Create main project directory
os.makedirs('sports_prediction_model', exist_ok=True)
os.chdir('sports_prediction_model')

# Create subdirectories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/external', exist_ok=True)
os.makedirs('notebooks', exist_ok=True)
os.makedirs('src/data', exist_ok=True)
os.makedirs('src/features', exist_ok=True)
os.makedirs('src/models', exist_ok=True)
os.makedirs('src/utils', exist_ok=True)
os.makedirs('config', exist_ok=True)
os.makedirs('tests', exist_ok=True)
os.makedirs('models/saved', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('reports/figures', exist_ok=True)

print("Project directory structure created.")

# Verify the structure (optional)
# for root, dirs, files in os.walk('.'):
#     level = root.replace('.', '').count(os.sep)
#     indent = ' ' * 4 * (level)
#     print(f'{indent}{os.path.basename(root)}/')
#     subindent = ' ' * 4 * (level + 1)
#     for f in files:
#         print(f'{subindent}{f}')

Project directory structure created.


In [ ]:
import os

# Define the content for requirements.txt
requirements_content = """
# Data manipulation
polars

# Visualization
matplotlib
seaborn
plotly

# Machine Learning
scikit-learn
xgboost
lightgbm
catboost

# Deep Learning
torch
tensorflow

# Bayesian/Statistical
pymc==5.9.0
statsmodels==0.14.0
pystan==3.8.0

# Sports data libraries
pybaseball
nba_api==1.3.0
statsbombpy==1.6.0
understat==0.1.0

# Utilities
tqdm==4.65.0
joblib==1.3.1
python-dotenv==1.0.0
loguru==0.7.2

# API/Web
requests==2.31.0
beautifulsoup4==4.12.2
selenium==4.11.2

# Jupyter
jupyter==1.0.0
ipykernel==6.25.0
"""

# Write requirements.txt to the current working directory
# This ensures it's written in 'sports_prediction_model/' after os.chdir
with open(os.path.join(os.getcwd(), 'requirements.txt'), 'w') as f:
    f.write(requirements_content)

print("requirements.txt created in current working directory.")

requirements.txt created in current working directory.


In [ ]:
# Install pyyaml separately to avoid build issues
!pip install pyyaml

In [ ]:
# Install remaining dependencies from requirements.txt, forcing no cache to avoid stale lookups
!pip install -r requirements.txt --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 49.3 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × installing build dependencies for pytensor did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
ERROR: Failed to build 'pytensor' when installing build dependencies for pytensor


In [ ]:
# This cell is now redundant after refactoring installation steps and will be ignored.

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install numpy pandas

In [ ]:
%%writefile config/config.yaml
---
project:
  name: "Sports Prediction Model"
  version: "1.0.0"
  author: "Your Name"
  description: "Multi-sport predictive modeling for baseball, basketball, soccer"

data:
  baseball:
    source: "baseball_savant"
    seasons: [2020, 2021, 2022, 2023, 2024]
    data_dir: "data/raw/baseball"
    processed_dir: "data/processed/baseball"

  basketball:
    source: "nba_api"
    seasons: ["2020-21", "2021-22", "2022-23", "2023-24"]
    data_dir: "data/raw/basketball"
    processed_dir: "data/processed/basketball"

  soccer:
    source: "statsbomb"
    competitions: [43, 44, 45]  # World Cup, Euros, etc.
    data_dir: "data/raw/soccer"
    processed_dir: "data/processed/soccer"

features:
  baseball:
    - launch_angle
    - exit_velocity
    - pitch_type
    - count
    - spray_angle
    - park_factor

  basketball:
    - shot_distance
    - defender_distance
    - shot_clock
    - quarter
    - score_differential

  soccer:
    - shot_location
    - body_part
    - assist_type
    - pressure
    - game_state

models:
  baseball:
    target: "hit_probability"
    algorithm: "xgboost"
    hyperparameters:
      n_estimators: 300
      max_depth: 6
      learning_rate: 0.05

  basketball:
    target: "points_next_possession"
    algorithm: "lightgbm"
    hyperparameters:
      n_estimators: 500
      num_leaves: 31
      learning_rate: 0.03

  soccer:
    target: "goal_probability"
    algorithm: "catboost"
    hyperparameters:
      iterations: 1000
      depth: 8
      learning_rate: 0.05

training:
  test_size: 0.2
  validation_size: 0.1
  random_state: 42
  cv_folds: 5

evaluation:
  metrics:
    - accuracy
    - precision
    - recall
    - f1
    - roc_auc
    - log_loss

In [ ]:
%%writefile src/data/baseball_loader.py
import pandas as pd
import numpy as np
from pybaseball import statcast, batting_stats, pitching_stats
from datetime import datetime, timedelta
import os
from loguru import logger
import time

class BaseballDataLoader:
    """
    Data loader for MLB/Baseball data from multiple sources
    """

    def __init__(self, config_path='config/config.yaml'):
        self.config = self._load_config(config_path)
        self.data_dir = self.config['data']['baseball']['data_dir']
        self.seasons = self.config['data']['baseball']['seasons']

    def _load_config(self, config_path):
        import yaml
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)

    def download_statcast_data(self, start_date, end_date, batch_size=30):
        """
        Download Statcast data in batches to handle large requests
        """
        start = datetime.strptime(start_date, '%Y-%m-%d')
        end = datetime.strptime(end_date, '%Y-%m-%d')

        all_data = []
        current = start

        while current < end:
            batch_end = min(current + timedelta(days=batch_size), end)
            logger.info(f"Downloading Statcast data: {current.date()} to {batch_end.date()}")

            try:
                data = statcast(start_dt=current.strftime('%Y-%m-%d'),
                              end_dt=batch_end.strftime('%Y-%m-%d'))
                if not data.empty:
                    all_data.append(data)
                time.sleep(2)  # Rate limiting
            except Exception as e:
                logger.error(f"Error downloading batch: {e}")

            current = batch_end + timedelta(days=1)

        if all_data:
            return pd.concat(all_data, ignore_index=True)
        return pd.DataFrame()

    def download_season_stats(self, year):
        """
        Download batting and pitching stats for a season
        """
        logger.info(f"Downloading {year} season stats")

        # Batting stats
        batters = batting_stats(year)
        batters['season'] = year
        batters['stat_type'] = 'batting'

        # Pitching stats
        pitchers = pitching_stats(year)
        pitchers['season'] = year
        pitchers['stat_type'] = 'pitching'

        return batters, pitchers

    def load_historical_data(self, seasons=None):
        """
        Load data for multiple seasons
        """
        if seasons is None:
            seasons = self.seasons

        all_batters = []
        all_pitchers = []

        for year in seasons:
            batters, pitchers = self.download_season_stats(year)
            all_batters.append(batters)
            all_pitchers.append(pitchers)
            time.sleep(1)

        return {
            'batters': pd.concat(all_batters, ignore_index=True),
            'pitchers': pd.concat(all_pitchers, ignore_index=True)
        }

    def save_raw_data(self, data, filename):
        """
        Save raw data to disk
        """
        os.makedirs(self.data_dir, exist_ok=True)
        filepath = os.path.join(self.data_dir, filename)

        if isinstance(data, dict):
            for key, df in data.items():
                df.to_csv(f"{filepath}_{key}.csv", index=False)
        else:
            data.to_csv(filepath, index=False)

        logger.info(f"Saved data to {filepath}")

# Example usage
if __name__ == "__main__":
    loader = BaseballDataLoader()

    # Download Statcast data for June 2024
    statcast_data = loader.download_statcast_data('2024-06-01', '2024-06-30')
    loader.save_raw_data(statcast_data, 'statcast_june2024.csv')

    # Download season stats
    historical_stats = loader.load_historical_data([2023, 2024])
    loader.save_raw_data(historical_stats, 'historical_stats')

In [ ]:
%%writefile src/data/data_pipeline.py
import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger
import json
from datetime import datetime

class DataPipeline:
    """
    Unified data pipeline for all sports
    """

    def __init__(self, sport='baseball', config_path='config/config.yaml'):
        self.sport = sport
        self.config = self._load_config(config_path)
        self.raw_dir = Path(f"data/raw/{sport}")
        self.processed_dir = Path(f"data/processed/{sport}")

        # Create directories
        self.raw_dir.mkdir(parents=True, exist_ok=True)
        self.processed_dir.mkdir(parents=True, exist_ok=True)

    def _load_config(self, config_path):
        import yaml
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)

    def load_raw_data(self, filename):
        """
        Load raw data from disk
        """
        filepath = self.raw_dir / filename
        if filepath.suffix == '.csv':
            return pd.read_csv(filepath)
        elif filepath.suffix == '.parquet':
            return pd.read_parquet(filepath)
        elif filepath.suffix == '.json':
            with open(filepath, 'r') as f:
                return json.load(f)
        else:
            raise ValueError(f"Unsupported file format: {filepath.suffix}")

    def save_processed_data(self, data, filename):
        """
        Save processed data
        """
        filepath = self.processed_dir / filename
        if filename.endswith('.csv'):
            data.to_csv(filepath, index=False)
        elif filename.endswith('.parquet'):
            data.to_parquet(filepath, index=False)
        logger.info(f"Saved processed data to {filepath}")

    def create_training_dataset(self, X, y, features, target):
        """
        Create training-ready dataset
        """
        # Ensure X and y are aligned
        assert len(X) == len(y), "X and y must have same length"

        # Create feature matrix
        X_df = pd.DataFrame(X, columns=features)
        y_df = pd.DataFrame(y, columns=[target])

        # Combine
        dataset = pd.concat([X_df, y_df], axis=1)

        # Add metadata
        dataset['created_date'] = datetime.now().isoformat()
        dataset['sport'] = self.sport

        return dataset

    def split_data(self, df, target_col, test_size=0.2, val_size=0.1, random_state=42):
        """
        Split data into train/val/test sets
        """
        from sklearn.model_selection import train_test_split

        # Separate features and target
        X = df.drop(columns=[target_col, 'created_date', 'sport'])
        y = df[target_col]

        # First split: separate test set
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )

        # Second split: separate validation from training
        val_ratio = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_ratio, random_state=random_state
        )

        logger.info(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

        return {
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test
        }

In [ ]:
%%writefile src/features/baseball_features.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from loguru import logger

class BaseballFeatureEngineer:
    """
    Feature engineering for baseball data
    """

    def __init__(self):
        self.label_encoders = {}
        self.scaler = StandardScaler()

    def create_pitch_features(self, df):
        """
        Create features from pitch-level data
        """
        features = df.copy()

        # 1. Count features
        features['balls_strikes'] = features['balls'].astype(str) + '-' + features['strikes'].astype(str)
        features['is_full_count'] = ((features['balls'] == 3) & (features['strikes'] == 2)).astype(int)
        features['is_two_strikes'] = (features['strikes'] == 2).astype(int)
        features['is_hitters_count'] = ((features['balls'] > features['strikes']) & (features['balls'] >= 2)).astype(int)
        features['is_pitchers_count'] = ((features['strikes'] > features['balls']) & (features['strikes'] >= 1)).astype(int)

        # 2. Pitch characteristics
        features['effective_speed'] = features['release_speed'] * (1 - 0.017 * (features['pfx_z'] / 12))
        features['horizontal_break'] = np.abs(features['pfx_x'])
        features['vertical_break'] = np.abs(features['pfx_z'])
        features['break_angle'] = np.arctan2(features['pfx_z'], features['pfx_x']) * 180 / np.pi
        features['break_magnitude'] = np.sqrt(features['pfx_x']**2 + features['pfx_z']**2)

        # 3. Location features
        features['plate_distance'] = np.sqrt(features['plate_x']**2 + features['plate_z']**2)
        features['is_center'] = ((np.abs(features['plate_x']) < 0.5) &
                                 (features['plate_z'] > 1.5) &
                                 (features['plate_z'] < 3.5)).astype(int)
        features['is_shadow'] = (((np.abs(features['plate_x']) < 0.5) &
                                  ((features['plate_z'] < 1.5) | (features['plate_z'] > 3.5))) |
                                 ((np.abs(features['plate_x']) >= 0.5) &
                                  (features['plate_z'] > 1.5) &
                                  (features['plate_z'] < 3.5))).astype(int)

        # 4. Batter-pitcher matchup features
        features['same_hand'] = (features['batter_hand'] == features['pitcher_hand']).astype(int)

        # 5. Game context features
        features['is_close_game'] = (np.abs(features['home_score'] - features['away_score']) <= 2).astype(int)
        features['high_leverage'] = ((features['outs'] >= 2) &
                                     (features['on_3b'].notna()) &
                                     (features['inning'] >= 7)).astype(int)

        # 6. Batter hotness (rolling averages)
        if 'batter_id' in features.columns and 'game_date' in features.columns:
            features = self._add_rolling_stats(features, 'batter_id', 'launch_speed', 10)
            features = self._add_rolling_stats(features, 'batter_id', 'launch_angle', 10)

        return features

    def _add_rolling_stats(self, df, group_col, value_col, window):
        """
        Add rolling averages for a given column
        """
        df = df.sort_values(['game_date', group_col])
        rolling_mean = df.groupby(group_col)[value_col].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        df[f'{value_col}_rolling_{window}'] = rolling_mean
        return df

    def encode_categorical(self, df, columns):
        """
        Encode categorical variables
        """
        for col in columns:
            if col in df.columns:
                if col not in self.label_encoders:
                    self.label_encoders[col] = LabelEncoder()
                    df[f'{col}_encoded'] = self.label_encoders[col].fit_transform(df[col].fillna('unknown'))
                else:
                    df[f'{col}_encoded'] = self.label_encoders[col].transform(df[col].fillna('unknown'))
        return df

    def normalize_features(self, df, feature_cols):
        """
        Normalize numerical features
        """
        # Select only numeric columns that exist
        cols_to_normalize = [col for col in feature_cols if col in df.columns and df[col].dtype in ['int64', 'float64']]

        if cols_to_normalize:
            df[cols_to_normalize] = self.scaler.fit_transform(df[cols_to_normalize])

        return df

    def prepare_for_modeling(self, df, target_col='events'):
        """
        Complete feature preparation pipeline
        """
        logger.info(f"Starting feature preparation. Shape: {df.shape}")

        # Create features
        df = self.create_pitch_features(df)

        # Define categorical columns
        cat_cols = ['pitch_type', 'balls_strikes', 'batter_hand', 'pitcher_hand']
        df = self.encode_categorical(df, cat_cols)

        # Define numerical features
        num_features = [
            'release_speed', 'release_spin', 'release_extension',
            'pfx_x', 'pfx_z', 'plate_x', 'plate_z',
            'effective_speed', 'horizontal_break', 'vertical_break',
            'break_angle', 'break_magnitude', 'plate_distance',
            'is_full_count', 'is_two_strikes', 'is_hitters_count',
            'is_pitchers_count', 'is_center', 'is_shadow',
            'same_hand', 'is_close_game', 'high_leverage'
        ]

        # Add rolling features if they exist
        rolling_cols = [col for col in df.columns if 'rolling' in col]
        num_features.extend(rolling_cols)

        # Normalize
        df = self.normalize_features(df, num_features)

        # Create target variable (example: hit probability)
        if target_col in df.columns:
            df['is_hit'] = df[target_col].isin(['single', 'double', 'triple', 'home_run']).astype(int)

        logger.info(f"Feature preparation complete. Final shape: {df.shape}")

        return df

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Ensure the base_model can be imported, assuming current directory is sports_prediction_model
# If not, sys.path modification or direct import path might be needed. For now, assume relative import works.
# Note: This is a temporary bypass for the import issue in a raw string, if it persists, it needs adjustment.
# For proper execution of this standalone cell, a simpler BaseModel stub or full path import would be better.
# However, the primary goal here is to fix the FileNotFoundError.

filepath = 'src/models/ensemble_model.py'
os.makedirs(os.path.dirname(filepath), exist_ok=True)

file_content = """
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from .base_model import BaseModel
from loguru import logger

class EnsembleModel(BaseModel):
    """
    Ensemble model combining multiple algorithms
    """

    def __init__(self, model_name="ensemble", sport="baseball", config=None):
        super().__init__(model_name, sport, config)
        self.models = {}
        self.weights = {}
        self.meta_model = None

    def build_model(self):
        """
        Build ensemble of models
        """
        # Individual models
        self.models = {
            'xgboost': XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                random_state=42,
                n_jobs=-1
            ),
            'lightgbm': LGBMClassifier(
                n_estimators=300,
                num_leaves=31,
                learning_rate=0.05,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            ),
            'catboost': CatBoostClassifier(
                iterations=300,
                depth=6,
                learning_rate=0.05,
                random_state=42,
                verbose=False
            ),
            'random_forest': RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            )
        }

        logger.info(f"Built ensemble with {len(self.models)} models")

        return self.models

    def train(self, X_train, y_train, X_val=None, y_val=None):
        """
        Train all models in ensemble
        """
        self.features = list(X_train.columns)

        # Store the number of classes from y_train
        self._num_classes = len(np.unique(y_train))

        # Train each model
        for name, model in self.models.items():
            logger.info(f"Training {name}...")
            model.fit(X_train, y_train)

            # Evaluate on validation set if provided
            if X_val is not None:
                score = model.score(X_val, y_val)
                logger.info(f"{name} validation score: {score:.4f}")
                self.weights[name] = score
            else:
                self.weights[name] = 1.0 / len(self.models)

        # Normalize weights
        total = sum(self.weights.values())
        self.weights = {k: v/total for k, v in self.weights.items()}

        logger.info(f"Ensemble weights: {self.weights}")

        return self.models

    def predict(self, X):
        """
        Weighted voting prediction
        """
        if not hasattr(self, '_num_classes'):
            raise RuntimeError("Model must be trained first to determine number of classes.")

        predictions = np.zeros((len(X), self._num_classes))

        for name, model in self.models.items():
            pred_proba = model.predict_proba(X)
            predictions += self.weights[name] * pred_proba

        return np.argmax(predictions, axis=1)

    def predict_proba(self, X):
        """
        Weighted probability prediction
        """
        # Assuming binary classification for now if _num_classes isn't set, as in original code
        num_cols = self._num_classes if hasattr(self, '_num_classes') else 2
        probabilities = np.zeros((len(X), num_cols))

        for name, model in self.models.items():
            proba = model.predict_proba(X)
            probabilities += self.weights[name] * proba

        return probabilities

    def feature_importance_aggregated(self):
        """
        Aggregate feature importance across models
        """
        importance_dict = {}

        for name, model in self.models.items():
            if hasattr(model, 'feature_importances_'):
                for i, imp in enumerate(model.feature_importances_):
                    feature = self.features[i]
                    if feature not in importance_dict:
                        importance_dict[feature] = []
                    importance_dict[feature].append(imp * self.weights[name])

        # Average importance
        aggregated = {
            feature: np.mean(imps)
            for feature, imps in importance_dict.items()
        }

        return pd.DataFrame(
            list(aggregated.items()),
            columns=['feature', 'importance']
        ).sort_values('importance', ascending=False)
"""

with open(filepath, 'w') as f:
    f.write(file_content)

print(f"Writing {filepath}")

Writing src/models/ensemble_model.py


FileNotFoundError: [Errno 2] No such file or directory: 'src/models/ensemble_model.py'

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger
import yaml
import json
from datetime import datetime
import sys
import textwrap # Import textwrap to dedent the string content

filepath = 'src/train_pipeline.py'
os.makedirs(os.path.dirname(filepath), exist_ok=True)

file_content = textwrap.dedent("""
    import pandas as pd
    import numpy as np
    from pathlib import Path
    from loguru import logger
    import yaml
    import json
    from datetime import datetime
    import sys

    # Add src to path
    sys.path.append(str(Path(__file__).parent.parent))

    from data.baseball_loader import BaseballDataLoader
    from data.data_pipeline import DataPipeline
    from features.baseball_features import BaseballFeatureEngineer
    from models.xgboost_model import XGBoostModel
    from models.ensemble_model import EnsembleModel

    class TrainingPipeline:
        """
        End-to-end training pipeline
        """

        def __init__(self, config_path='config/config.yaml', sport='baseball'):
            self.config = self._load_config(config_path)
            self.sport = sport
            self.data_pipeline = DataPipeline(sport=sport, config_path=config_path)
            self.feature_engineer = BaseballFeatureEngineer()  # Sport-specific
            self.model = None

            # Setup logging
            logger.add(f"logs/training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

        def _load_config(self, config_path):
            with open(config_path, 'r') as f:
                return yaml.safe_load(f)

        def prepare_data(self, raw_data_path=None):
            """
            Prepare data for training
            """
            logger.info("Starting data preparation...")

            # Load raw data
            if raw_data_path:
                df = self.data_pipeline.load_raw_data(raw_data_path)
            else:
                # Download fresh data
                loader = BaseballDataLoader()
                df = loader.download_statcast_data('2024-06-01', '2024-06-30')

            # Engineer features
            df_processed = self.feature_engineer.prepare_for_modeling(df)

            # Save processed data
            self.data_pipeline.save_processed_data(
                df_processed,
                f'processed_{datetime.now().strftime("%Y%m%d")}.csv'
            )

            logger.info(f"Data preparation complete. Shape: {df_processed.shape}")

            return df_processed

        def train_model(self, df, target_col='is_hit', model_type='xgboost'):
            """
            Train the model
            """
            logger.info(f"Starting model training with {model_type}...")

            # Split data
            feature_cols = [col for col in df.columns if col not in [target_col, 'events', 'game_date']]
            X = df[feature_cols]
            y = df[target_col]

            # Remove columns with NaN
            X = X.dropna(axis=1, how='any')

            # Split into train/val/test
            split_data = self.data_pipeline.split_data(
                pd.concat([X, y], axis=1),
                target_col
            )

            # Initialize model
            if model_type == 'xgboost':
                model_config = self.config['models']['baseball']
                self.model = XGBoostModel(
                    model_name="xgboost_baseball",
                    sport=self.sport,
                    config=model_config
                )
            elif model_type == 'ensemble':
                self.model = EnsembleModel(
                    model_name="ensemble_baseball",
                    sport=self.sport
                )
            else:
                raise ValueError(f"Unknown model type: {model_type}")

            # Build model
            self.model.build_model()

            # Train
            self.model.train(
                split_data['X_train'],
                split_data['y_train'],
                split_data['X_val'],
                split_data['y_val']
            )

            # Evaluate
            metrics = self.model.evaluate(split_data['X_test'], split_data['y_test'])
            logger.info(f"Test metrics: {metrics}")

            # Feature importance
            if hasattr(self.model, 'feature_importance'):
                importance = self.model.feature_importance()
                logger.info(f"Top 10 features:\n{importance.head(10)}")

            return self.model, metrics

        def optimize_model(self, df, target_col='is_hit', n_trials=50):
            """
            Optimize hyperparameters
            """
            logger.info("Starting hyperparameter optimization...")

            # Prepare data
            feature_cols = [col for col in df.columns if col not in [target_col, 'events', 'game_date']]
            X = df[feature_cols].dropna(axis=1, how='any')
            y = df[target_col]

            # Split
            split_data = self.data_pipeline.split_data(
                pd.concat([X, y], axis=1),
                target_col
            )

            # Initialize model
            self.model = XGBoostModel(sport=self.sport)
            self.model.build_model()

            # Optimize
            best_params = self.model.optimize_hyperparameters(
                split_data['X_train'],
                split_data['y_train'],
                split_data['X_val'],
                split_data['y_val'],
                n_trials=n_trials
            )

            # Save best params
            with open(f'models/best_params_{datetime.now().strftime("%Y%m%d")}.json', 'w') as f:
                json.dump(best_params, f, indent=2)

            logger.info(f"Optimization complete. Best params: {best_params}")

            return best_params

        def save_model(self, filename=None):
            """
            Save trained model
            """
            if self.model is None:
                raise ValueError("No model to save")

            if filename is None:
                filename = f'models/saved/{self.sport}_{self.model.model_name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl'

            self.model.save_model(filename)
            return filename

        def run_complete_pipeline(self, raw_data_path=None, model_type='xgboost'):
            """
            Run the complete training pipeline
            """
            logger.info("="*50)
            logger.info("Starting complete training pipeline")
            logger.info("="*50)

            # Step 1: Prepare data
            df = self.prepare_data(raw_data_path)

            # Step 2: Train model
            model, metrics = self.train_model(df, model_type=model_type)

            # Step 3: Save model
            model_path = self.save_model()

            # Step 4: Save metrics
            metrics_path = f'reports/metrics_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
            with open(metrics_path, 'w') as f:
                json.dump(metrics, f, indent=2, default=str)

            logger.info("="*50)
            logger.info("Pipeline complete!")
            logger.info(f"Model saved to: {model_path}")
            logger.info(f"Metrics saved to: {metrics_path}")
            logger.info("="*50)

            return model, metrics

    # Example usage
    if __name__ == "__main__":
        # Initialize pipeline
        pipeline = TrainingPipeline(sport='baseball')

        # Run complete pipeline
        model, metrics = pipeline.run_complete_pipeline(
            raw_data_path=None,  # Set to None to download fresh data
            model_type='xgboost'
        )

        # Print feature importance
        if hasattr(model, 'feature_importance'):
            print("\nTop 10 Features:")
            print(model.feature_importance().head(10))
    """)

with open(filepath, 'w') as f:
    f.write(file_content)

print(f"Writing {filepath}")

IndentationError: unexpected indent (3184054411.py, line 35)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
import textwrap # Import textwrap to dedent the string content
warnings.filterwarnings('ignore')

filepath = 'src/evaluation/dashboard.py'
os.makedirs(os.path.dirname(filepath), exist_ok=True)

file_content = textwrap.dedent('''
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    import json
    from pathlib import Path
    import warnings
    warnings.filterwarnings('ignore')

    class ModelDashboard:
        """
        Interactive dashboard for model evaluation
        """

        def __init__(self, model, X_test, y_test, feature_names=None):
            self.model = model
            self.X_test = X_test
            self.y_test = y_test
            self.feature_names = feature_names or X_test.columns.tolist()
            self.y_pred = model.predict(X_test)

            if hasattr(model, 'predict_proba'):
                self.y_proba = model.predict_proba(X_test)

        def plot_confusion_matrix(self, save_path=None):
            """
            Plot confusion matrix
            """
            from sklearn.metrics import confusion_matrix

            cm = confusion_matrix(self.y_test, self.y_pred)

            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                       xticklabels=['No Hit', 'Hit'],
                       yticklabels=['No Hit', 'Hit'])
            plt.title('Confusion Matrix')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')

            if save_path:
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

        def plot_roc_curve(self, save_path=None):
            """
            Plot ROC curve
            """
            from sklearn.metrics import roc_curve, auc

            if not hasattr(self.model, 'predict_proba'):
                print("Model doesn't support probability predictions")
                return

            fpr, tpr, _ = roc_curve(self.y_test, self.y_proba[:, 1])
            roc_auc = auc(fpr, tpr)

            plt.figure(figsize=(8, 6))
            plt.plot(fpr, tpr, color='darkorange', lw=2,
                    label=f'ROC curve (AUC = {roc_auc:.2f})')
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('Receiver Operating Characteristic (ROC) Curve')
            plt.legend(loc="lower right")

            if save_path:
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

        def plot_feature_importance(self, top_n=20, save_path=None):
            """
            Plot feature importance
            """
            if not hasattr(self.model, 'feature_importance'):
                print("Model doesn't support feature importance")
                return

            importance = self.model.feature_importance()
            importance = importance.head(top_n)

            plt.figure(figsize=(10, 8))
            bars = plt.barh(range(len(importance)), importance['importance'].values)
            plt.yticks(range(len(importance)), importance['feature'].values)
            plt.xlabel('Importance')
            plt.title(f'Top {top_n} Feature Importance')

            # Add value labels
            for i, (bar, val) in enumerate(zip(bars, importance['importance'].values)):
                plt.text(val, i, f'{val:.3f}', va='center')

            if save_path:
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

        def plot_calibration_curve(self, n_bins=10, save_path=None):
            """
            Plot calibration curve
            """
            from sklearn.calibration import calibration_curve

            if not hasattr(self.model, 'predict_proba'):
                print("Model doesn't support probability predictions")
                return

            prob_true, prob_pred = calibration_curve(
                self.y_test, self.y_proba[:, 1], n_bins=n_bins
            )

            plt.figure(figsize=(8, 6))
            plt.plot(prob_pred, prob_true, marker='o', linewidth=2,
                    label='Model')
            plt.plot([0, 1], [0, 1], linestyle='--', color='gray',
                    label='Perfectly Calibrated')
            plt.xlabel('Mean Predicted Probability')
            plt.ylabel('Fraction of Positives')
            plt.title('Calibration Curve')
            plt.legend()

            if save_path:
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

        def plot_prediction_distribution(self, save_path=None):
            """
            Plot distribution of predictions vs actual
            """
            if not hasattr(self.model, 'predict_proba'):
                print("Model doesn't support probability predictions")
                return

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # Actual distribution
            axes[0].hist(self.y_test, bins=2, alpha=0.7, color='blue', edgecolor='black')
            axes[0].set_title('Actual Distribution')
            axes[0].set_xlabel('Class')
            axes[0].set_ylabel('Frequency')
            axes[0].set_xticks([0, 1])

            # Predicted probabilities distribution
            axes[1].hist(self.y_proba[:, 1], bins=20, alpha=0.7, color='green', edgecolor='black')
            axes[1].set_title('Predicted Probability Distribution')
            axes[1].set_xlabel('Predicted Probability of Hit')
            axes[1].set_ylabel('Frequency')

            plt.tight_layout()

            if save_path:
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.show()

        def create_interactive_dashboard(self, output_file='dashboard.html'):
            """
            Create interactive Plotly dashboard
            """
            # Create subplots
            fig = make_subplots(
                rows=2, cols=2,
                subplot_titles=('ROC Curve', 'Feature Importance',
                              'Calibration Curve', 'Prediction Distribution'),
                specs=[[{'type': 'scatter'}, {'type': 'bar'}],
                       [{'type': 'scatter'}, {'type': 'histogram'}]]
            )

            # ROC Curve
            from sklearn.metrics import roc_curve, auc
            fpr, tpr, _ = roc_curve(self.y_test, self.y_proba[:, 1])
            roc_auc = auc(fpr, tpr)

            fig.add_trace(
                go.Scatter(x=fpr, y=tpr, mode='lines',
                          name=f'ROC (AUC={roc_auc:.3f})',
                          line=dict(color='darkorange', width=2)),
                row=1, col=1
            )
            fig.add_trace(
                go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                          name='Random', line=dict(color='navy', dash='dash')),n                row=1, col=1
            )

            # Feature Importance
            if hasattr(self.model, 'feature_importance'):
                importance = self.model.feature_importance().head(15)
                fig.add_trace(
                    go.Bar(x=importance['importance'], y=importance['feature'],
                          orientation='h', name='Importance',
                          marker_color='lightblue'),
                    row=1, col=2
                )

            # Calibration Curve
            from sklearn.calibration import calibration_curve
            prob_true, prob_pred = calibration_curve(self.y_test, self.y_proba[:, 1], n_bins=10)

            fig.add_trace(
                go.Scatter(x=prob_pred, y=prob_true, mode='lines+markers',
                          name='Model', line=dict(color='green', width=2)),
                row=2, col=1
            )
            fig.add_trace(
                go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                          name='Perfect', line=dict(color='gray', dash='dash')),n                row=2, col=1
            )

            # Prediction Distribution
            fig.add_trace(
                go.Histogram(x=self.y_proba[:, 1], nbinsx=20,
                            name='Probabilities', marker_color='purple'),
                row=2, col=2
            )

            # Update layout
            fig.update_layout(
                height=800,
                showlegend=True,
                title_text="Model Performance Dashboard",
                title_font_size=20
            )

            fig.update_xaxes(title_text="False Positive Rate", row=1, col=1)
            fig.update_yaxes(title_text="True Positive Rate", row=1, col=1)

            fig.update_xaxes(title_text="Importance", row=1, col=2)
            fig.update_yaxes(title_text="Feature", row=1, col=2)

            fig.update_xaxes(title_text="Mean Predicted Probability", row=2, col=1)
            fig.update_yaxes(title_text="Fraction of Positives", row=2, col=1)

            fig.update_xaxes(title_text="Predicted Probability", row=2, col=2)
            fig.update_yaxes(title_text="Frequency", row=2, col=2)

            # Save
            fig.write_html(output_file)
            print(f"Dashboard saved to {output_file}")

            return fig

        def generate_report(self, output_dir='reports'):
            """
            Generate comprehensive evaluation report
            """
            from sklearn.metrics import classification_report, confusion_matrix

            Path(output_dir).mkdir(parents=True, exist_ok=True)

            # Classification report
            report = classification_report(self.y_test, self.y_pred, output_dict=True)

            # Confusion matrix
            cm = confusion_matrix(self.y_test, self.y_pred)

            # Create report
            report_data = {
                'classification_report': report,
                'confusion_matrix': cm.tolist(),
                'accuracy': (cm[0,0] + cm[1,1]) / cm.sum(),
                'n_samples': len(self.y_test)
            }

            # Add ROC AUC if available
            if hasattr(self.model, 'predict_proba'):
                from sklearn.metrics import roc_auc_score
                report_data['roc_auc'] = roc_auc_score(self.y_test, self.y_proba[:, 1])

            # Save report
            report_path = Path(output_dir) / 'evaluation_report.json'
            with open(report_path, 'w') as f:
                json.dump(report_data, f, indent=2, default=str)

            # Generate plots
            self.plot_confusion_matrix(Path(output_dir) / 'confusion_matrix.png')
            self.plot_roc_curve(Path(output_dir) / 'roc_curve.png')
            self.plot_calibration_curve(Path(output_dir) / 'calibration.png')
            self.plot_feature_importance(Path(output_dir) / 'feature_importance.png')

            print(f"Report generated in {output_dir}")

            return report_data

    # Example usage
    if __name__ == "__main__":
        # Load model and test data (example)
        # dashboard = ModelDashboard(model, X_test, y_test)
        # dashboard.create_interactive_dashboard()
        # dashboard.generate_report()
        pass
    ''')

with open(filepath, 'w') as f:
    f.write(file_content)

print(f"Writing {filepath}")

Writing src/evaluation/dashboard.py


In [ ]:
!python main.py --sport baseball --action train --model-type xgboost

Traceback (most recent call last):
  File "/content/main.py", line 9, in <module>
    from loguru import logger
ModuleNotFoundError: No module named 'loguru'


In [ ]:
!python main.py --sport baseball --action train --model-type ensemble

Traceback (most recent call last):
  File "/content/main.py", line 9, in <module>
    from loguru import logger
ModuleNotFoundError: No module named 'loguru'


In [ ]:
!python main.py --sport baseball --action train --data-path data/raw/statcast_june2024.csv

Traceback (most recent call last):
  File "/content/main.py", line 9, in <module>
    from loguru import logger
ModuleNotFoundError: No module named 'loguru'


In [ ]:
%%writefile main.py
#!/usr/bin/env python3
"""
Main entry point for sports prediction model
"""

import argparse
import sys
from pathlib import Path
from loguru import logger

# Add src to path
sys.path.append(str(Path(__file__).parent))

from src.train_pipeline import TrainingPipeline
from src.models.xgboost_model import XGBoostModel
from src.models.ensemble_model import EnsembleModel
from src.evaluation.dashboard import ModelDashboard

def parse_args():
    parser = argparse.ArgumentParser(description='Sports Prediction Model')

    parser.add_argument('--sport', type=str, default='baseball',
                       choices=['baseball', 'basketball', 'soccer'],
                       help='Sport to model')

    parser.add_argument('--action', type=str, default='train',
                       choices=['train', 'optimize', 'evaluate', 'predict'],
                       help='Action to perform')

    parser.add_argument('--model-type', type=str, default='xgboost',
                       choices=['xgboost', 'ensemble'],
                       help='Type of model to use')

    parser.add_argument('--data-path', type=str, default=None,
                       help='Path to raw data file')

    parser.add_argument('--model-path', type=str, default=None,
                       help='Path to saved model')

    parser.add_argument('--config', type=str, default='config/config.yaml',
                       help='Path to config file')

    parser.add_argument('--optimize-trials', type=int, default=50,
                       help='Number of optimization trials')

    return parser.parse_args()

def main():
    args = parse_args()

    logger.info(f"Starting {args.sport} prediction model")
    logger.info(f"Action: {args.action}")

    if args.action == 'train':
        # Initialize pipeline
        pipeline = TrainingPipeline(
            config_path=args.config,
            sport=args.sport
        )

        # Run training
        model, metrics = pipeline.run_complete_pipeline(
            raw_data_path=args.data_path,
            model_type=args.model_type
        )

        logger.success("Training complete!")

    elif args.action == 'optimize':
        # Initialize pipeline
        pipeline = TrainingPipeline(
            config_path=args.config,
            sport=args.sport
        )

        # Load data
        if args.data_path:
            df = pipeline.data_pipeline.load_raw_data(args.data_path)
        else:
            logger.error("Data path required for optimization")
            return

        # Run optimization
        best_params = pipeline.optimize_model(
            df,
            n_trials=args.optimize_trials
        )

        logger.success(f"Optimization complete! Best params: {best_params}")

    elif args.action == 'evaluate':
        if not args.model_path or not args.data_path:
            logger.error("Model path and data path required for evaluation")
            return

        # Load model
        model = XGBoostModel()
        model.load_model(args.model_path)

        # Load test data
        import pandas as pd
        df = pd.read_csv(args.data_path)

        # Prepare features
        feature_cols = [col for col in df.columns if col not in ['is_hit', 'events']]
        X_test = df[feature_cols]
        y_test = df['is_hit']

        # Create dashboard
        dashboard = ModelDashboard(model, X_test, y_test)

        # Generate report
        dashboard.generate_report('reports/evaluation')

        # Create interactive dashboard
        dashboard.create_interactive_dashboard('reports/dashboard.html')

        logger.success("Evaluation complete! Check reports/ directory")

    elif args.action == 'predict':
        if not args.model_path:
            logger.error("Model path required for prediction")
            return

        # Load model
        model = XGBoostModel()
        model.load_model(args.model_path)

        # Load data to predict
        if args.data_path:
            import pandas as pd
            df = pd.read_csv(args.data_path)

            # Make predictions
            predictions = model.predict(df)
            probabilities = model.predict_proba(df)

            # Save predictions
            results = df.copy()
            results['prediction'] = predictions
            results['probability'] = probabilities[:, 1]

            output_path = 'predictions.csv'
            results.to_csv(output_path, index=False)

            logger.success(f"Predictions saved to {output_path}")
        else:
            # Interactive prediction
            logger.info("Enter feature values (or press Ctrl+C to exit):")
            # Add interactive input logic here
            pass

if __name__ == "__main__":
    main()

Writing main.py
